<a href="https://colab.research.google.com/github/TWalkerSMCM/icsa-scraper/blob/main/examples/export-skipper-finishes.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# icsa-scraper · Export Skipper Finishes to CSV

A utility notebook: dump **every skipper's finish in every fleet race** of a
season (or several) to a single flat CSV — one row per skipper per race, with
full regatta context on each row. Feed it to a spreadsheet, R, a database, or
any rating experiment that lives outside Python.

Deliberately unfiltered beyond `boat_role == "skipper"`: women's and coed
regattas, every boat class, penalty rows, and the occasional blank
`sailor_slug` (an RP entry the site never resolved to a profile) are all
included — filter to taste downstream, or borrow the filter cells from
`skipper-elo.ipynb` / `skipper-bradley-terry.ipynb`.

## Install

In [ ]:
# Fresh runtimes (Colab) need the library. Guard on the distribution name so
# re-running locally can't clobber an editable dev install; to force-update a
# cached Colab runtime, restart it (or run the %pip line with --force-reinstall).
from importlib.metadata import PackageNotFoundError, version

try:
    version("icsa-scraper")
except PackageNotFoundError:
    %pip install -q "icsa-scraper[fetch] @ git+https://github.com/TWalkerSMCM/icsa-scraper"

## Optional: persist the cache on Colab

`scraper.load()` caches every fetched page under `./.scraper_cache`, so re-running a cell (or the whole notebook) after the first pass is instant. Colab wipes that folder on runtime reset — mount Drive and point the cache there first if you want it to survive. Set `SCRAPER_CACHE_DIR` **before** importing `scraper`:

```python
# from google.colab import drive
# drive.mount('/content/drive')
# import os
# os.environ['SCRAPER_CACHE_DIR'] = '/content/drive/MyDrive/icsa_cache'
```

## What to export

Swap the seasons for any codes on the site (`f` fall / `s` spring + 2-digit
year). The output filename tracks them automatically.

In [ ]:
# The full academic year: fall + spring.
SEASONS = ["f25", "s26"]

OUT = f"skipper-finishes-{'-'.join(SEASONS)}.csv"

COLUMNS = [
    "season",
    "regatta_slug",
    "regatta_name",
    "start_time",
    "regatta_type",
    "boat",
    "participant",  # "coed" | "women" (inferred from the page; see the library docs)
    "division",
    "race_num",
    "sailor_slug",  # stable cross-regatta identity; "" when the site never resolved one
    "sailor_name",
    "school_slug",
    "team_name",
    "place",  # scored finish, penalties included in the points
    "penalty",  # e.g. "DNF", "DSQ"; blank for a clean sailed finish
]

## Load and flatten

`.fleet()` narrows the dataset to fleet-scoring regattas only;
`progress=True` shows a per-regatta progress bar while it runs.
`sailor_races_frame()` is already the flat RP↔finish join — one row per
sailor per race, pre-joined with regatta context. All this notebook does is
keep the skipper rows and order them chronologically.

In [ ]:
import scraper

data = scraper.load(SEASONS, sailors=True, progress=True).fleet()

df = data.sailor_races_frame()
skippers = (
    df[df["boat_role"] == "skipper"][COLUMNS]
    .sort_values(["start_time", "regatta_slug", "division", "race_num", "place"])
    .reset_index(drop=True)
)
print(f"{len(skippers):,} skipper finishes across {len(data.regattas)} fleet regattas")
skippers.head()

## Write the CSV

In [ ]:
import os

skippers.to_csv(OUT, index=False)
print(f"{OUT}: {len(skippers):,} rows, {os.path.getsize(OUT) / 1e6:.1f} MB")

## Getting the file out of Colab

Locally the CSV lands next to the notebook and you're done. On Colab the file
lives on the runtime's ephemeral disk — download it or copy it to Drive:

```python
# from google.colab import files
# files.download(OUT)

# ...or, with Drive mounted (see the cache tip above):
# import shutil; shutil.copy(OUT, "/content/drive/MyDrive/" + OUT)
```

---
**Variations:** drop the `boat_role` filter to export crews too (or export
`data.finishes_frame()` for per-team rather than per-sailor rows);
`df[df["participant"] == "women"]` for the women's circuit only;
`df[df["penalty"].notna()]` for a penalty audit. For a per-regatta rather than
per-race grain, `data.results_frame()` is one row per school per regatta.